# Possession: LightGBM trained on IDSSE, applied to BAR-ATH

**Pipeline.** Clean external tracking (IDSSE, 7 matches) → degraded to look like the BAR-ATH ATD →
per-frame features → LightGBM (H / A / D) → Viterbi decoder (switch penalties) → 199,319-row CSV.

**How we choose anything.** Leave-one-match-out on IDSSE with the organisers' metric code
(`solution/evaluate_external.py`). The public BAR-ATH 5 minutes are used **once, at the end, as an
alignment check** — never to pick features, penalties or label variants (RULES.md).

**Reproducibility.** The final file is regenerated by `python -m solution.submission ...` (last
cell prints the exact command). This notebook calls the same functions.

Sections that need the BAR-ATH data (§2, §3 calibration, §7–§10) skip themselves until the bundle
is unpacked into `./data`.

In [ ]:
import os, sys, json, time, subprocess
ROOT = os.path.abspath('..')
for p in (ROOT, os.path.join(ROOT, 'starter'), os.path.join(ROOT, 'evaluation')):
    if p not in sys.path:
        sys.path.insert(0, p)
os.chdir(ROOT)  # every path below is relative to the repository root

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from solution import (atd, container as C, dataset_fit, decode, degrade, evaluate_external as ev,
                      features, idsse, labels, submission)
pm = C.pm

TEAM = 'MyTeam'      # the file is written as evaluation/results/<TEAM>.csv
SEED = 0
FULL = True          # False: 3 matches per harness run, ~3x faster, noisier -- for iterating only
MATCHES = None if FULL else ['J03WMX', 'J03WN1', 'J03WOH']
QUIET = lambda s: None
HAVE_DATA = atd.have_data()
print('BAR-ATH data present:', HAVE_DATA)

## 1. External data: why IDSSE

Seven DFL matches (2022/23, TRACAB, 25 Hz, CC-BY 4.0). Same inputs as the ATD — ~22 players with a
team, ball (x, y), 105 × 68 m — and **labels built by the same rule as the challenge ground truth**:
every frame carries `BallStatus` (alive/dead) and `BallPossession` (owning team), so
`D if dead else owner` is the GT recipe from CHALLENGE.md verbatim.

The known difference: IDSSE's owner flag changes hands about twice as often as the public window
(possessions per alive minute). The last row is shown to compare **definitions**, nothing is fitted
to it.

In [ ]:
ext = idsse.load_all()
inputs = pd.DataFrame([dict(match=m['name'], **dataset_fit.input_stats(m)) for m in ext])
lab = pd.DataFrame([dict(match=m['name'], **dataset_fit.label_stats(m['label'])) for m in ext])
public = pm.label_codes(pm.read_label_csv(dataset_fit.PUBLIC_GT))
lab.loc[len(lab)] = dict(match='BAR-ATH public 5 min', **dataset_fit.label_stats(public))
display(inputs[['match', 'players_mean', 'slots', 'ball_seen', 'pitch']])
display(lab.round(2))

## 2. BAR-ATH tactical ATD → the same container

`solution/atd.py` parses the raw ATD (format in `starter/baseline.py`), converts 0–1 coordinates
to metres and marks the two graded halves. The checks recompute DATA.md's published figures; a
mismatch means the load is wrong, so stop there.

In [ ]:
bar = None
if HAVE_DATA:
    bar = atd.load()
    bar_checks = atd.checks(bar)
else:
    print('skipped: unpack the match bundle into ./data (DATA.md)')

## 3. Calibrate the degradation on the real ATD

Training and validation run on IDSSE made to look like the ATD. The defaults are DATA.md's
published figures (ball missing 41 %, 20.5 players). With the data present, the burst lengths are
**measured on BAR-ATH** — statistics of the inputs, never of a label, so this is allowed.

Sanity check that matters: the organisers' baseline scores **spellF1 0.2433 / macroF1 0.6724** on
the real match. On degraded IDSSE it should land near those numbers; if macroF1 stays far lower,
the real ATD probably loses the ball mostly *during stoppages*, and `w_off` / a dead-ball weight
should go up (see `degrade.DEFAULTS`).

In [ ]:
DEGRADE_PARAMS = {}
if HAVE_DATA:
    s = degrade.atd_noise_stats(bar)
    print('BAR-ATH ATD   ', {k: round(v, 3) for k, v in s.items()})
    DEGRADE_PARAMS = {'ball_missing': s['ball_missing'], 'ball_run_s': s['ball_run_s_mean'],
                      'player_visible': s['players_visible_mean'] / 22}
probe = degrade.degrade(ext[0], seed=SEED, **DEGRADE_PARAMS)
print('degraded IDSSE', {k: round(v, 3) for k, v in degrade.atd_noise_stats(probe).items()})
os.makedirs('data/models', exist_ok=True)
json.dump(DEGRADE_PARAMS, open('data/models/degrade_params.json', 'w'))

def run_lengths(m):
    miss = np.isnan(m['bx']) & (m['half'] > 0)
    e = np.diff(np.concatenate(([0], miss.astype(np.int8), [0])))
    return (np.flatnonzero(e == -1) - np.flatnonzero(e == 1)) / C.FPS

fig, ax = plt.subplots(figsize=(8, 3))
bins = np.logspace(-1.6, 2, 40)
ax.hist(run_lengths(probe), bins=bins, alpha=.6, density=True, label='degraded IDSSE')
if HAVE_DATA:
    ax.hist(run_lengths(bar), bins=bins, alpha=.6, density=True, label='BAR-ATH ATD')
ax.set_xscale('log'); ax.set_xlabel('missing-ball burst (s)'); ax.legend(); plt.show()
del probe

## 4. Feature engineering

**v0** (`features.build_features`, 112 columns) — only inputs both datasets have, oriented so home
attacks +x, missing ball left as NaN for LightGBM:

| group | features |
|---|---|
| ball | seen, position, speed, distance outside / to the lines |
| player ↔ ball | nearest home / away distance and their difference, players within 3 m |
| players only | per-team speed, centroid progress toward goal and its velocity, spread, length, width |
| occlusion | frames since / until the ball is seen, and what it showed then (nearest team, out of play) |
| temporal | centred ±0.2 / 1 / 3 s and past / future 1 / 4 s means of the key series |

**v1** (`features.build_features_v1`, 203 columns) = v0 + four blocks, each one idea:

| block | idea | source |
|---|---|---|
| `contact_block` | ball changes direction / speed within 1.5 m of a player → a touch; last / next touching team | Vidal-Codina et al. 2022 |
| `streak_block` | how long the same team has been nearest the ball | control ≈ persistence |
| `deadball_block` | players beyond the lines, at a corner, kickoff layout; where the ball was heading when lost; 10 s stillness | restart triggers, Lang et al. 2022 |
| `pressure_block` | players of each team within 5 m | duels vs control |

Each harness run: 7 folds, ~4–7 min.

In [ ]:
res = {}
res['v0'] = ev.run(MATCHES, feature_fn=features.build_features, degrade_params=DEGRADE_PARAMS,
                   seed=SEED, log=QUIET)
ev.report(res['v0'], 'v0')

In [ ]:
res['v1'] = ev.run(MATCHES, feature_fn=features.build_features_v1, degrade_params=DEGRADE_PARAMS,
                   seed=SEED, log=QUIET)
ev.report(res['v1'], 'v1')

### v0 vs v1, match by match

A gain counts only if it is consistent across held-out matches and larger than the pooled
bootstrap interval — the same discipline the organisers apply to the board.

In [ ]:
def compare(a, b):
    df = pd.DataFrame({'match': [r['match'] for r in res[a]['rows']],
                       a: [r['spellF1'] for r in res[a]['rows']],
                       b: [r['spellF1'] for r in res[b]['rows']]})
    df['diff'] = df[b] - df[a]
    display(df.round(4))
    for k in (a, b):
        p = res[k]['pooled']
        print('{:<4} pooled spellF1 {:.4f}  [{:.4f} .. {:.4f}]  macroF1 {:.4f}  possessions {} / {}'.format(
            k, p['spell_f1_mean'], p['spell_bootstrap_low'], p['spell_bootstrap_high'], p['macro_f1'],
            p['n_pred_spells'], p['n_gt_spells']))
    print('{} better on {}/{} matches, mean diff {:+.4f}'.format(b, (df['diff'] > 0).sum(), len(df), df['diff'].mean()))
    return df

cmp_v = compare('v0', 'v1')
FEATURES = 'v1' if (cmp_v['diff'] > 0).sum() > len(cmp_v) / 2 and cmp_v['diff'].mean() > 0 else 'v0'
FEATURE_FN = features.FEATURE_SETS[FEATURES]
print('-> using', FEATURES)

In [ ]:
gain = pd.Series(res[FEATURES]['gain'], index=res[FEATURES]['names']).sort_values(ascending=False)
gain.head(30)[::-1].plot.barh(figsize=(7, 8), title='LightGBM gain, ' + FEATURES); plt.show()

### Feature lab

Write one block, run the harness on base + block, keep it only if it wins as above. `ctx` holds the
oriented arrays in metres (`px, py` (T, S), `bx, by`, `home`, `away` masks, `dist` player-ball
(T, S), `seen`, `L`, `W`); `f` holds the features built so far. Return `(new_series, temporal_keys)`.

In [ ]:
def my_block(ctx, f):
    # example: distance from the ball to the goal the home team attacks
    return {'ball_to_home_target': np.hypot(ctx['L'] - ctx['bx'], ctx['by'] - ctx['W'] / 2)}, []

RUN_LAB = False
if RUN_LAB:
    base = features.V1_BLOCKS if FEATURES == 'v1' else ()
    res['lab'] = ev.run(MATCHES, feature_fn=features.with_blocks(*base, my_block),
                        degrade_params=DEGRADE_PARAMS, seed=SEED, log=QUIET)
    ev.report(res['lab'], 'lab')
    compare(FEATURES, 'lab')

## 5. Label definition: raw IDSSE vs "touches are not possessions"

CHALLENGE.md: a touch without control or a deflection is **not** a change of possession.
`labels.merge_sandwiched(tau)` absorbs an opposing spell shorter than `tau` between two spells of
the same team. `tau` is set from that sentence (0.5 s), not tuned — the public window itself keeps
0.8 s and 1.3 s sandwiched spells, so `tau` has to stay below that.

The two variants have *different* ground truths, so their own scores are not comparable. Instead
each model is scored against **both** definitions; prefer the one that is robust to not knowing
which definition Metrica uses (best worst case).

In [ ]:
TAU = 0.5
res['tau'] = ev.run(MATCHES, feature_fn=FEATURE_FN, label_fn=labels.harmonised(TAU),
                    degrade_params=DEGRADE_PARAMS, seed=SEED, log=QUIET)

raw_by_name = {m['name']: m['label'] for m in ext}
def pooled_vs(r, definition):
    gts, prs = [], []
    sep = np.full(C.HALF_GAP, C.X, np.int8)
    for d, pred in zip(r['data'], r['preds']):
        gt = raw_by_name[d['m']['name'].rsplit('_deg', 1)[0]]
        gt = labels.merge_sandwiched(gt, TAU) if definition == 'tau' else gt
        gts += [gt, sep]; prs += [pred, sep]
    return pm.spell_scores(np.concatenate(gts), np.concatenate(prs), pm.valid_runs(np.concatenate(gts)))['mean_f1']

table = pd.DataFrame({'scored vs raw': [pooled_vs(res[FEATURES], 'raw'), pooled_vs(res['tau'], 'raw')],
                      'scored vs tau=%.1f' % TAU: [pooled_vs(res[FEATURES], 'tau'), pooled_vs(res['tau'], 'tau')]},
                     index=['trained on raw', 'trained on tau=%.1f' % TAU])
table['worst case'] = table.min(1)
display(table.round(4))
LABEL_TAU = 0.0 if table['worst case'].iloc[0] >= table['worst case'].iloc[1] else TAU
LABEL_FN = labels.harmonised(LABEL_TAU) if LABEL_TAU else labels.raw
PAIR = res['tau' if LABEL_TAU else FEATURES]['best_pair']
print('-> training labels: tau =', LABEL_TAU, '| decoder penalties (team/dead):', PAIR)

## 6. Final model: all seven matches, two noise draws, both orientations

Same function as `python -m solution.submission`, so the notebook and the CLI produce the same model.

In [ ]:
for k in list(res):            # free memory before the big fit
    res[k].pop('data', None); res[k].pop('probs', None)
t0 = time.time()
model = submission.train_final(FEATURE_FN, LABEL_FN, DEGRADE_PARAMS, log=QUIET)
model.save_model('data/models/{}.txt'.format(TEAM))
print('trained in {:.0f}s, {} trees'.format(time.time() - t0, model.num_trees()))

## 7. Predict BAR-ATH

No extra degradation here — the ATD is already the noisy thing the model was trained to read.

In [ ]:
prob = None
if HAVE_DATA:
    prob = model.predict(FEATURE_FN(bar)[0])
    a, b = 11185, 11185 + 25 * 300   # first 5 minutes of play, for a look
    t = (np.arange(a, b) - a) / 25
    fig, ax = plt.subplots(figsize=(13, 3))
    for k, name in enumerate('HAD'):
        ax.plot(t, prob[a:b, k], lw=.8, label=name)
    ax.fill_between(t, 0, 1, where=np.isnan(bar['bx'][a:b]), color='grey', alpha=.15, label='ball missing')
    ax.set_xlabel('s'); ax.legend(ncol=4); plt.show()
else:
    print('skipped: no BAR-ATH data')

## 8. Smart Tagging prior (optional, BAR-ATH only)

IDSSE has no Smart Tagging, so it cannot be learned — it enters as a small log-prior on the tagged
team (`submission.smart_tagging_prior`). Its weight is **not** tuned on the public window. The
check below is unsupervised: on frames where the model is already confident, how often does the tag
agree? High agreement → the tag is a reliable tie-breaker for the unconfident frames.

Assumption to verify by watching a few clips: on a *defending* phase, `Team` is the defending side.

In [ ]:
SMART_TAGGING_BETA = 0.0       # off until the agreement check justifies it
if HAVE_DATA:
    unit = submission.smart_tagging_prior(len(prob), beta=1.0)
    tagged = unit[:, :2].max(1) > 0
    tag_side = np.where(unit[:, 0] > unit[:, 1], C.H, C.A)
    model_side = prob.argmax(1)
    sel = tagged & (prob.max(1) > 0.8) & (model_side != C.D) & (bar['half'] > 0)
    print('tag agrees with a confident model on {:.1%} of {} frames'.format(
        np.mean(tag_side[sel] == model_side[sel]), sel.sum()))
    if SMART_TAGGING_BETA:
        prob = submission.apply_prior(prob, submission.smart_tagging_prior(len(prob), SMART_TAGGING_BETA))

## 9. Decode and write the submission

In [ ]:
if HAVE_DATA:
    codes = submission.to_codes(prob, bar['half'], PAIR)
    path = os.path.join('evaluation', 'results', TEAM + '.csv')
    digest = submission.write_csv(codes, path)
    print(submission.summary(codes))
    print(path, 'sha256', digest)

## 10. Checks

1. `validate_submission.py` — the exact checks the official scorer runs.
2. Possession count and mean duration vs the IDSSE ranges in §1 — a count far outside them means
   the penalties do not transfer.
3. `run_evaluation.py` on the public window — **alignment check only** (are transitions on the right
   frames, is H Barcelona). Do not change anything because of this number: 17 possessions, ±6 points
   per possession, and the rules forbid it.

In [ ]:
if HAVE_DATA:
    subprocess.run([sys.executable, 'evaluation/validate_submission.py', path], check=False)
    subprocess.run([sys.executable, 'evaluation/run_evaluation.py'], check=False)

## 11. Reproduce from the command line

This must regenerate the identical file (same sha256) — RULES.md §3.

In [ ]:
cmd = [sys.executable, '-m', 'solution.submission', '--team', TEAM, '--features', FEATURES,
       '--tau', str(LABEL_TAU), '--pair', str(PAIR[0]), str(PAIR[1]),
       '--degrade-json', 'data/models/degrade_params.json',
       '--smart-tagging', str(SMART_TAGGING_BETA)]
print(' '.join(cmd[1:]))
RUN_REPRO = False
if HAVE_DATA and RUN_REPRO:
    out = subprocess.run(cmd, capture_output=True, text=True).stdout
    print(out)
    print('identical:', digest in out)